In [9]:
import google.generativeai as genai
import os

def get_api_key(filepath="apikey.txt"):
    """指定されたファイルからAPIキーを読み込む"""
    try:
        with open(filepath, "r") as f:
            return f.read().strip()
    except FileNotFoundError:
        print(f"エラー: APIキーファイル '{filepath}' が見つかりません。")
        return None

def solve_historical_ordering_via_gemini(api_key, events):
    """
    Gemini APIを使用して歴史的な出来事を年代順に並べ替える。

    Args:
        api_key (str): Gemini APIキー。
        events (dict): キーが出来事の記号 (例: 'ア')、値が出来事の説明文の辞書。

    Returns:
        list: 年代順に並べられた出来事の記号のリスト。
               エラーが発生した場合はNone。
    """
    if not api_key:
        return None

    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-2.0-flash-lite') # または 'gemini-1.5-pro-latest' など

    event_dates = {}

    for symbol, description in events.items():
        prompt = f"""
        以下の日本の歴史上の出来事が発生したおおよその西暦年を答えてください。
        数字のみで答えてください。例えば「810」のように。

        出来事: {description}
        """
        try:
            print(f"\nGeminiに問い合わせ中: {symbol} - {description[:30]}...")
            response = model.generate_content(prompt)
            # APIからのレスポンスのテキスト部分から年を抽出する
            # より堅牢なエラー処理や形式チェックが必要になる場合があります
            year_text = response.text.strip()
            if year_text.isdigit():
                event_dates[symbol] = int(year_text)
                print(f"  {symbol}: {year_text}年")
            else:
                print(f"  エラー: {symbol} の年を取得できませんでした。レスポンス: {year_text}")
                # 念のため、レスポンス全体を出力してデバッグしやすくする
                # print(f"  Full response for {symbol}: {response}")
                # 年が取得できない場合はエラーとして扱うか、デフォルト値を設定するかなどを検討
                event_dates[symbol] = float('inf') # 並び替えで最後にくるように
        except Exception as e:
            print(f"  API呼び出し中にエラーが発生しました ({symbol}): {e}")
            # エラーが発生した場合の処理 (例: Noneを返す、例外を再スローするなど)
            event_dates[symbol] = float('inf') # 並び替えで最後にくるように

    # 年に基づいて出来事をソート
    # event_datesに有効な年が入っているか確認
    if not all(isinstance(year, int) or year == float('inf') for year in event_dates.values()):
        print("\nエラー: 全ての出来事の年を正しく取得できませんでした。")
        return None

    # 年が取得できなかった項目を除外するか、エラー処理をここで行う
    valid_event_dates = {k: v for k, v in event_dates.items() if v != float('inf')}
    if len(valid_event_dates) != len(events):
        print("\n警告: 一部の出来事の年を特定できませんでした。結果が不正確な可能性があります。")

    # 辞書を値（年）でソートし、キー（記号）のリストを取得
    try:
        sorted_symbols = sorted(valid_event_dates, key=valid_event_dates.get)
    except Exception as e:
        print(f"\n並び替え中にエラーが発生しました: {e}")
        return None

    return sorted_symbols

if __name__ == "__main__":
    # 問題の定義
    problem_description = """
9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。
"""
    events_to_sort = {
        "ア": "藤原時平は，策謀を用いて菅原道真を政界から追放した。",
        "イ": "嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。",
        "ウ": "藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。"
    }

    print("問題:")
    print(problem_description)

    # APIキーの取得
    api_key = get_api_key()

    if api_key:
        print("\nGemini APIを使用して解答を生成します...")
        # Gemini APIを呼び出して問題を解く
        ordered_events = solve_historical_ordering_via_gemini(api_key, events_to_sort)

        if ordered_events:
            print("\n解答 (年代の古い順):")
            print(" → ".join(ordered_events))
        else:
            print("\n解答の生成に失敗しました。")
    else:
        print("APIキーが設定されていないため、処理を中止します。")

問題:

9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。


Gemini APIを使用して解答を生成します...

Geminiに問い合わせ中: ア - 藤原時平は，策謀を用いて菅原道真を政界から追放した。...
  ア: 901年

Geminiに問い合わせ中: イ - 嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。...
  イ: 810年

Geminiに問い合わせ中: ウ - 藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。...
  ウ: 842年

解答 (年代の古い順):
イ → ウ → ア


In [7]:
import os
import itertools
from google import genai

def main():
    # Read the Gemini API key from apikey.txt
    script_dir = os.path.dirname(os.path.realpath(__file__))
    key_path = os.path.join(script_dir, "apikey.txt")
    with open(key_path, "r") as f:
        api_key = f.read().strip()

    # Configure the Gemini client
    genai.configure(api_key=api_key)

    # Define the historical statements
    statements = {
        "ア": "藤原時平は，策謀を用いて菅原道真を政界から追放した。",
        "イ": "嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。",
        "ウ": "藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。"
    }

    # Generate all possible orderings of the keys
    orderings = [" → ".join(order) for order in itertools.permutations(statements.keys(), 3)]

    # Construct the prompt
    statement_desc = "\n".join(f"{k}: {v}" for k, v in statements.items())
    choices_desc = "\n".join(f"{i+1}. {order}" for i, order in enumerate(orderings))
    prompt = (
        "以下の3つの歴史的出来事を年代の古い順に並べてください。\n\n"
        f"{statement_desc}\n\n"
        "選択肢:\n"
        f"{choices_desc}\n\n"
        "正しい選択肢の番号と並び順を「番号: 並び順」の形式で答えてください。"
    )

    # Call the Gemini Chat API
    response = genai.chat.create(
        model="gemini-2.0-flash-lite",  # 利用する Gemini モデル
        messages=[
            {"role": "system", "content": "歴史的事実の年代順を判断してください。"},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
    )

    # Print the result
    print("API Response:\n", response.choices[0].message.content)

if __name__ == "__main__":
    main()


NameError: name '__file__' is not defined

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import sys

def check_dependencies():
    missing = []
    try:
        import torch
    except ImportError:
        missing.append("torch")
    try:
        import transformers
    except ImportError:
        missing.append("transformers")
    if missing:
        print("Error: 以下のライブラリが見つかりませんでした：", ", ".join(missing))
        print("実行前に次のコマンドでインストールしてください：")
        print("  pip install " + " ".join(missing))
        sys.exit(1)

def main():
    # 依存チェック
    check_dependencies()

    # transformers と permutations を読み込む
    from transformers import pipeline
    from itertools import permutations

    # zero-shot 分類器の初期化
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

    # 問題の文言
    statements = {
        "ア": "藤原時平は，策謀を用いて菅原道真を政界から追放した。",
        "イ": "嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。",
        "ウ": "藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。"
    }

    # 文をまとめたシーケンス
    sequence = " ".join(f"{k}: {v}" for k, v in statements.items())

    # 全順列の候補ラベルを作成
    candidates = [" → ".join(p) for p in permutations(statements.keys(), 3)]

    # zero-shot 推論
    result = classifier(sequence, candidate_labels=candidates)

    # 最上位ラベルが解答
    print("Predicted chronological order:", result["labels"][0])

if __name__ == "__main__":
    main()



RuntimeError: At least one of TensorFlow 2.0 or PyTorch should be installed. To install TensorFlow 2.0, read the instructions at https://www.tensorflow.org/install/ To install PyTorch, read the instructions at https://pytorch.org/.

In [2]:
from transformers import pipeline
from itertools import permutations

# Initialize the zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define the three statements from the problem
statements = {
    "ア": "藤原時平は，策謀を用いて菅原道真を政界から追放した。",
    "イ": "嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。",
    "ウ": "藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。"
}

# Prepare the combined sequence
sequence = " ".join(f"{key}: {text}" for key, text in statements.items())

# Generate all possible orderings
orderings = [" → ".join(order) for order in permutations(statements.keys(), 3)]

# Zero-shot classification: treat each ordering as a candidate label
result = classifier(sequence, orderings)

# Display the predicted ordering
print("Predicted correct chronological order:")
print(result['labels'][0])


c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\eriya\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk.

RuntimeError: At least one of TensorFlow 2.0 or PyTorch should be installed. To install TensorFlow 2.0, read the instructions at https://www.tensorflow.org/install/ To install PyTorch, read the instructions at https://pytorch.org/.